# HW7 Lab: Build a RAG System 📚🔍🤖

<!-- lab-header:begin -->
**MIS 769 · Big Data Analytics for Business · Fall 2026**

**Points:** 40  |  **Due:** Sunday, October 25, 2026 at 10:00 PM Pacific

**Dr. Richard J. Young** · UNLV Lee Business School
<!-- lab-header:end -->

> **Why are we learning this?** LLMs know the public internet up to their training
> cutoff — they know *nothing* about YOUR company's return policy, YOUR contracts, or
> last week's meeting notes. RAG (Retrieval-Augmented Generation) fixes that: retrieve
> the relevant documents, paste them into the prompt, let the model answer from them.
> It is the single most deployed LLM pattern in business today. The transferable skill:
> **connecting an LLM to private knowledge without retraining anything.**

**How this lab works:** everything below is a *working demo* — hit Runtime → Run all
and a complete RAG system builds and answers questions on a fictional company's
documents. Your job: (1) understand each stage, (2) twist the knobs in the 🧪
Experiment cells, and (3) rebuild the corpus from **a domain you're passionate about**
in the 🎮 Playground — where YOU can judge whether the answers are right.

**The pipeline:** Query → Retrieve context (vector search) → Augment prompt → Generate

```
Query ──▶ Embed ──▶ Search vector store ──▶ top-k chunks ──▶ "Context: … Question: …" ──▶ LLM ──▶ Answer
```

⚠️ **Do NOT use pre-built services like Google NotebookLM for this assignment.** The
point is to BUILD the pipeline yourself so you understand every moving part.


## Learning objectives

- Split long documents into overlapping chunks and reason about the size/overlap trade-off
- Embed documents and build a FAISS vector index for similarity search
- Implement retrieval (`retrieve_similar`) and a full RAG pipeline (`rag_answer`)
- Compare RAG answers vs. pure-LLM answers and explain when RAG wins


## Setup

Three models/libraries, all free and local: **sentence-transformers** to embed text,
**FAISS** to search vectors fast, and **flan-t5-base** (an *instruction-tuned* model —
it actually follows "answer the question from this context") to generate answers.


In [1]:
# dependencies
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")   # 384-dim embeddings

# flan-t5 is a seq2seq (encoder-decoder) model. transformers v5 removed the
# "text2text-generation" pipeline task, and routing T5 through "text-generation"
# runs but returns garbage, so we call the seq2seq classes directly.
_tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
_mod = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generator(prompt, max_new_tokens=80, **params):
    """Same shape the text2text-generation pipeline returned: [{"generated_text": ...}]."""
    ids = _tok(prompt, return_tensors="pt", truncation=True, max_length=512)
    out = _mod.generate(**ids, max_new_tokens=max_new_tokens, **params)
    return [{"generated_text": _tok.decode(out[0], skip_special_tokens=True)}]

print("✅ Embedding model + generator loaded")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 02:14:00.879000 17527 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 26424.65it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Embedding model + generator loaded


## Part 1: Document Chunking (6 pts)

Real documents are too long to embed whole: one embedding for a 5-page policy averages
away every detail, and stuffing whole documents into the prompt blows the context
window. So we **chunk**: split into pieces small enough to embed precisely, with
**overlap** so a sentence that straddles a boundary isn't cut in half and lost.

- **Small chunks** → precise retrieval, but each hit carries little context.
- **Large chunks** → more context per hit, but retrieval gets fuzzy and you waste prompt space.
- **Overlap** → insurance against boundary cuts; costs some storage duplication.

Here's a longer policy document to practice on:


In [2]:
policy_document = """TechGadgets Inc. Customer Policy Handbook.
Returns and Refunds. Customers may return most items within 30 days of delivery for a
full refund. Electronics must be returned within 14 days and must include all original
packaging and accessories. Refunds are issued to the original payment method within 5-7
business days after we receive the item. Opened software, gift cards, and clearance
items marked final sale cannot be returned. To start a return, log into your account
and select the order, or contact support for a prepaid shipping label.
Shipping. Standard shipping is free on orders over 50 dollars within the United States
and takes 3-5 business days. Express shipping costs 12.99 dollars and arrives in 1-2
business days. We ship internationally to over 50 countries; international orders may
be subject to customs duties which are the customer's responsibility. Orders placed
before 2pm Pacific ship the same business day.
Warranty. All electronics carry a 2-year manufacturer warranty covering defects in
materials and workmanship. The warranty does not cover accidental damage, water damage,
or normal wear. Extended protection plans are available at checkout for 1 to 3
additional years. Warranty claims require proof of purchase.
Accounts and Discounts. Students receive a 20 percent discount with a valid student ID
verified through our partner service. Premium members pay 29.99 dollars per month and
receive priority support, early access to sales, and free express shipping. Enterprise
customers receive dedicated account managers and volume pricing on orders over 100
units. Loyalty points are earned at 1 point per dollar and never expire."""

print(f"Document length: {len(policy_document.split())} words")

Document length: 256 words


`chunk_text` below is the workhorse: word-based chunks of `chunk_size` words,
where consecutive chunks share `overlap` words. Every RAG framework (LangChain,
LlamaIndex…) ships a fancier version of exactly this function.


In [3]:
def chunk_text(text, chunk_size=60, overlap=15):
    """Split text into overlapping word-based chunks."""
    words = text.split()
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks

chunks = chunk_text(policy_document, chunk_size=60, overlap=15)
print(f"{len(chunks)} chunks of ~60 words (overlap 15)")
print("\nFirst chunk:\n", chunks[0])

6 chunks of ~60 words (overlap 15)

First chunk:
 TechGadgets Inc. Customer Policy Handbook. Returns and Refunds. Customers may return most items within 30 days of delivery for a full refund. Electronics must be returned within 14 days and must include all original packaging and accessories. Refunds are issued to the original payment method within 5-7 business days after we receive the item. Opened software, gift cards, and clearance


In [4]:
# Feel the trade-off: try three settings and eyeball what a 'chunk' contains.
for size, ov in [(30, 5), (60, 15), (150, 30)]:
    cs = chunk_text(policy_document, chunk_size=size, overlap=ov)
    print(f"chunk_size={size:3d} overlap={ov:2d} → {len(cs):2d} chunks | "
          f"e.g. '{cs[len(cs)//2][:70]}...'")

chunk_size= 30 overlap= 5 → 11 chunks | e.g. 'orders may be subject to customs duties which are the customer's respo...'
chunk_size= 60 overlap=15 →  6 chunks | e.g. 'customer's responsibility. Orders placed before 2pm Pacific ship the s...'
chunk_size=150 overlap=30 →  2 chunks | e.g. 'to over 50 countries; international orders may be subject to customs d...'


### 🧪 Experiment

- Set `overlap=0` and print the chunk boundaries — find a sentence that got cut in half. That lost sentence is a question your RAG system can no longer answer.
- Try `chunk_size=30, overlap=5` and read a middle chunk out loud: could it answer "what does the warranty NOT cover?" on its own? Keep this intuition — you'll watch it break retrieval in Part 4.
- Push overlap to `chunk_size // 2`. How many chunks now? Overlap is insurance, and this is its storage premium.


In [5]:
# part 1 experiment 1
chunks_no_overlap = chunk_text(policy_document, chunk_size=60, overlap=0)
print(f"{len(chunks_no_overlap)} chunks with overlap=0\n")

# Show the end of each chunk and the start of the next one
for i in range(len(chunks_no_overlap) - 1):
    print(f"--- Boundary {i} -> {i + 1} ---")
    print("END OF CHUNK:   ...", " ".join(chunks_no_overlap[i].split()[-8:]))
    print("START OF NEXT:  ", " ".join(chunks_no_overlap[i + 1].split()[:8]), "...")
    print()

5 chunks with overlap=0

--- Boundary 0 -> 1 ---
END OF CHUNK:   ... the item. Opened software, gift cards, and clearance
START OF NEXT:   items marked final sale cannot be returned. To ...

--- Boundary 1 -> 2 ---
END OF CHUNK:   ... arrives in 1-2 business days. We ship internationally
START OF NEXT:   to over 50 countries; international orders may be ...

--- Boundary 2 -> 3 ---
END OF CHUNK:   ... wear. Extended protection plans are available at checkout
START OF NEXT:   for 1 to 3 additional years. Warranty claims ...

--- Boundary 3 -> 4 ---
END OF CHUNK:   ... receive dedicated account managers and volume pricing on
START OF NEXT:   orders over 100 units. Loyalty points are earned ...



In [6]:
# part 1 experiment 2
small_chunks = chunk_text(policy_document, chunk_size=30, overlap=5)
print(f"{len(small_chunks)} chunks of ~30 words (overlap 5)\n")

mid = len(small_chunks) // 2
print(f"Middle chunk (index {mid}):\n{small_chunks[mid]}\n")

# Where does the "does not cover" sentence actually live?
for i, c in enumerate(small_chunks):
    if "does not cover" in c:
        print(f"'does not cover' appears in chunk {i}:\n{c}")

# Can the middle chunk answer the warranty question on its own?
print("\nMiddle chunk contains 'does not cover'?", "does not cover" in small_chunks[mid])

11 chunks of ~30 words (overlap 5)

Middle chunk (index 5):
orders may be subject to customs duties which are the customer's responsibility. Orders placed before 2pm Pacific ship the same business day. Warranty. All electronics carry a 2-year manufacturer warranty

'does not cover' appears in chunk 6:
carry a 2-year manufacturer warranty covering defects in materials and workmanship. The warranty does not cover accidental damage, water damage, or normal wear. Extended protection plans are available at checkout

Middle chunk contains 'does not cover'? False


In [7]:
# part 1 experiment 3
total_words = len(policy_document.split())
print(f"Document length: {total_words} words\n")

for size in [30, 60]:
    print(f"chunk_size={size}")
    for ov in [0, size // 4, size // 2]:
        cs = chunk_text(policy_document, chunk_size=size, overlap=ov)
        stored = sum(len(c.split()) for c in cs)
        print(f"  overlap={ov:2d} -> {len(cs):2d} chunks | {stored} words stored "
              f"({stored / total_words:.2f}x the original)")
    print()

Document length: 256 words

chunk_size=30
  overlap= 0 ->  9 chunks | 256 words stored (1.00x the original)
  overlap= 7 -> 11 chunks | 326 words stored (1.27x the original)
  overlap=15 -> 17 chunks | 496 words stored (1.94x the original)

chunk_size=60
  overlap= 0 ->  5 chunks | 256 words stored (1.00x the original)
  overlap=15 ->  6 chunks | 331 words stored (1.29x the original)
  overlap=30 ->  8 chunks | 466 words stored (1.82x the original)



**In a markdown cell:** which setting would you pick for this document, and why?
(You'll defend this choice in Q1. Hint: could a 30-word chunk answer "what does the
warranty NOT cover" on its own?)


Answer: I would set the chunking=60 because it is in between the two extreme bounds and 15 words overlap because this is the length for the average sentence usually in most good technical docs. 

## Part 2: Embedding Index (8 pts)

Our knowledge base = a 12-entry company FAQ **plus** the policy chunks from Part 1.
We embed every piece into a 384-dimensional vector and load them into a **FAISS**
index (`IndexFlatL2` = exact nearest-neighbor search by L2 distance — brute force,
perfect at this scale).


In [8]:
# Company FAQ knowledge base
documents = [
    "Our company was founded in 2010 and is headquartered in San Francisco.",
    "We offer a 30-day money-back guarantee on all products.",
    "Customer support is available 24/7 via chat, email, or phone.",
    "Shipping is free for orders over $50 within the United States.",
    "Our premium subscription costs $29.99 per month and includes priority support.",
    "Returns must be initiated within 14 days of delivery.",
    "We accept Visa, Mastercard, American Express, and PayPal.",
    "International shipping is available to over 50 countries.",
    "Our mobile app is available for iOS and Android devices.",
    "Enterprise customers receive dedicated account managers.",
    "The warranty period for electronics is 2 years from purchase date.",
    "Student discounts of 20% are available with valid ID.",
]

# Combine the FAQ entries with the policy chunks into one corpus
corpus = documents + chunks
print(f"Corpus: {len(documents)} FAQ entries + {len(chunks)} policy chunks = {len(corpus)} pieces")

Corpus: 12 FAQ entries + 6 policy chunks = 18 pieces


In [9]:
corpus_embeddings = embedding_model.encode(corpus)
print(f"Embedding shape: {corpus_embeddings.shape}")
print(f"→ each piece of text is a {corpus_embeddings.shape[1]}-dimensional vector")

dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)          # exact L2-distance search
index.add(corpus_embeddings.astype("float32"))
print(f"✅ FAISS vector store built with {index.ntotal} vectors")

Embedding shape: (18, 384)
→ each piece of text is a 384-dimensional vector
✅ FAISS vector store built with 18 vectors


### 🧪 Experiment

- Embed two paraphrases — `embedding_model.encode(["full refund", "money-back guarantee"])` — and compute `np.linalg.norm(a - b)` between them. Now compare against two unrelated sentences. That distance gap is the entire magic of semantic search.
- Append one absurd off-topic document ("The moon is made of green cheese.") to `corpus`, rebuild the index, and see in Part 3 whether any reasonable question ever retrieves it.


In [10]:
# part 2 experiment 1
paraphrase = embedding_model.encode(["full refund", "money-back guarantee"])
dist_para = np.linalg.norm(paraphrase[0] - paraphrase[1])

# Unrelated pair: different words AND different meaning
unrel = embedding_model.encode(["The moon is made of green cheese.",
                                "How do I change a flat tire?"])
dist_unrel = np.linalg.norm(unrel[0] - unrel[1])

# Cross check: a refund phrase vs. an unrelated sentence
cross = embedding_model.encode(["full refund", "The moon is made of green cheese."])
dist_cross = np.linalg.norm(cross[0] - cross[1])

print(f"Paraphrases   ('full refund' vs 'money-back guarantee'):  L2 = {dist_para:.3f}")
print(f"Unrelated     ('green cheese' vs 'flat tire'):            L2 = {dist_unrel:.3f}")
print(f"Cross         ('full refund' vs 'green cheese'):          L2 = {dist_cross:.3f}")
print(f"\nGap (unrelated - paraphrase): {dist_unrel - dist_para:.3f}")

Paraphrases   ('full refund' vs 'money-back guarantee'):  L2 = 1.047
Unrelated     ('green cheese' vs 'flat tire'):            L2 = 1.376
Cross         ('full refund' vs 'green cheese'):          L2 = 1.407

Gap (unrelated - paraphrase): 0.329


In [11]:
# part 2 experiment 2
junk = "The moon is made of green cheese."

# Guard so re-running this cell doesn't append the junk doc twice
if junk not in corpus:
    corpus.append(junk)

# Rebuild embeddings + FAISS index over the updated corpus (same steps as the Part 2 index cell)
corpus_embeddings = embedding_model.encode(corpus)
index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
index.add(corpus_embeddings.astype("float32"))

print(f"Corpus now has {len(corpus)} pieces; FAISS index has {index.ntotal} vectors")
print(f"Junk doc is at index {corpus.index(junk)}")

# Sanity check: the junk doc should be retrievable by a question that actually matches it.
# (Uses index.search directly, since retrieve_similar isn't defined until Part 3.)
q = embedding_model.encode(["Is the moon made of cheese?"]).astype("float32")
dist, idx = index.search(q, 1)
print(f"\nTop hit for 'Is the moon made of cheese?': (L2={dist[0][0]:.3f}) {corpus[idx[0][0]]}")

Corpus now has 19 pieces; FAISS index has 19 vectors
Junk doc is at index 18

Top hit for 'Is the moon made of cheese?': (L2=0.314) The moon is made of green cheese.


## Part 3: Retrieval (10 pts)

Retrieval = embed the question with the **same** model, find the k nearest document
vectors. Lower L2 distance = more similar. This is the make-or-break step of every
RAG system: if the right document isn't in the top-k, the LLM can't answer correctly
no matter how smart it is.


In [12]:
def retrieve_similar(query, k=3):
    """Retrieve the k most similar corpus pieces for a query."""
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding.astype("float32"), k)
    return [{"document": corpus[idx], "score": float(distances[0][i]), "index": int(idx)}
            for i, idx in enumerate(indices[0])]

results = retrieve_similar("What is your return policy?", k=3)
for i, r in enumerate(results, 1):
    print(f"{i}. (L2={r['score']:.3f}) {r['document'][:90]}")

1. (L2=0.906) Returns must be initiated within 14 days of delivery.
2. (L2=1.054) TechGadgets Inc. Customer Policy Handbook. Returns and Refunds. Customers may return most 
3. (L2=1.059) We offer a 30-day money-back guarantee on all products.


In [13]:
# Probe retrieval quality — including questions designed to confuse it.
probe_queries = [
    "How long do I have to send back a laptop?",   # paraphrase, no shared keywords
    "Do you take PayPal?",
    "What does the warranty not cover?",
    "What time do you close on Sundays?",          # NOT in the corpus — watch what happens
]
for q in probe_queries:
    top = retrieve_similar(q, k=2)
    print(f"\nQ: {q}")
    for r in top:
        print(f"   (L2={r['score']:.3f}) {r['document'][:80]}")


Q: How long do I have to send back a laptop?
   (L2=0.894) TechGadgets Inc. Customer Policy Handbook. Returns and Refunds. Customers may re
   (L2=0.995) Returns must be initiated within 14 days of delivery.

Q: Do you take PayPal?
   (L2=0.876) We accept Visa, Mastercard, American Express, and PayPal.
   (L2=1.395) within 5-7 business days after we receive the item. Opened software, gift cards,

Q: What does the warranty not cover?
   (L2=0.823) customer's responsibility. Orders placed before 2pm Pacific ship the same busine
   (L2=0.991) The warranty period for electronics is 2 years from purchase date.

Q: What time do you close on Sundays?
   (L2=1.448) Returns must be initiated within 14 days of delivery.
   (L2=1.528) Customer support is available 24/7 via chat, email, or phone.


**Notice the last query:** the corpus says nothing about store hours, but FAISS
*always* returns the k nearest neighbors — nearest can still be far. Real systems set a
distance threshold. Keep a failure example like this for **Q3**.


### 🧪 Experiment

- Re-run the probes with `k=1` and then `k=10`. At k=1, how often is the single hit the right one? At k=10, how much junk rides along?
- Compare the L2 scores of a good hit ("Do you take PayPal?") vs the store-hours miss. Propose a numeric threshold that would separate them — then find a legitimate question your threshold would wrongly reject.
- Ask the same question with typos ("wat is ur retrun polcy"). Does semantic search shrug it off where keyword search would face-plant?


In [14]:
# part 3 experiment 1

probes = {
    "How long do I have to send back a laptop?": "14 days",
    "Do you take PayPal?": "PayPal",
    "What does the warranty not cover?": "does not cover",
    "What time do you close on Sundays?": None,
}

def is_relevant(doc, key):
    return key is not None and key.lower() in doc.lower()

for k in [1, 10]:
    print(f"{'=' * 70}\nk = {k}\n{'=' * 70}")
    hits = 0
    for q, key in probes.items():
        top = retrieve_similar(q, k=k)
        rel = [is_relevant(r["document"], key) for r in top]
        print(f"\nQ: {q}")
        for r, ok in zip(top, rel):
            mark = "✅" if ok else "  "
            print(f"  {mark} (L2={r['score']:.3f}) {r['document'][:70]}")
        if k == 1:
            hits += rel[0]
        else:
            n_rel = sum(rel)
            print(f"  → {n_rel} relevant, {k - n_rel} junk out of {k}")
    if k == 1:
        print(f"\nk=1: right top hit on {hits}/3 answerable questions "
              f"(store hours has no right answer)")

k = 1

Q: How long do I have to send back a laptop?
  ✅ (L2=0.894) TechGadgets Inc. Customer Policy Handbook. Returns and Refunds. Custom

Q: Do you take PayPal?
  ✅ (L2=0.876) We accept Visa, Mastercard, American Express, and PayPal.

Q: What does the warranty not cover?
  ✅ (L2=0.823) customer's responsibility. Orders placed before 2pm Pacific ship the s

Q: What time do you close on Sundays?
     (L2=1.448) Returns must be initiated within 14 days of delivery.

k=1: right top hit on 3/3 answerable questions (store hours has no right answer)
k = 10

Q: How long do I have to send back a laptop?
  ✅ (L2=0.894) TechGadgets Inc. Customer Policy Handbook. Returns and Refunds. Custom
  ✅ (L2=0.995) Returns must be initiated within 14 days of delivery.
     (L2=1.010) within 5-7 business days after we receive the item. Opened software, g
     (L2=1.170) The warranty period for electronics is 2 years from purchase date.
     (L2=1.238) customer's responsibility. Orders placed before 2pm Paci

In [15]:
# part 3 experiment 2
def best_score(q):
    """L2 distance of the single nearest corpus piece (lower = closer)."""
    return retrieve_similar(q, k=1)[0]["score"]

good = best_score("Do you take PayPal?")
miss = best_score("What time do you close on Sundays?")
print(f"Good hit  ('Do you take PayPal?'):               L2 = {good:.3f}")
print(f"Miss      ('What time do you close on Sundays?'): L2 = {miss:.3f}")

# Propose a threshold: midpoint between the good hit and the miss
threshold = (good + miss) / 2
print(f"\nProposed threshold: reject if best L2 > {threshold:.3f}\n")

# (question, is the answer actually in the corpus?)
tests = [
    ("How do I start a return?",              True),
    ("Do you have a mobile app?",             True),
    ("Is there a loyalty program?",           True),
    ("When was the company founded?",         True),
    ("Refund?",                               True),   # terse but legitimate
    ("How can I reach support at night?",     True),   # paraphrase of the 24/7 FAQ
    ("Who is the CEO?",                       False),
    ("What is the weather in Paris?",         False),
    ("What time do you close on Sundays?",    False),
]

for q, answerable in tests:
    s = best_score(q)
    accepted = s <= threshold
    if answerable and not accepted:
        verdict = "❌ WRONGLY REJECTED"
    elif not answerable and accepted:
        verdict = "❌ WRONGLY ACCEPTED"
    else:
        verdict = "✅ correct"
    print(f"(L2={s:.3f}) {'accept' if accepted else 'reject'}  {verdict:<20} {q}")

Good hit  ('Do you take PayPal?'):               L2 = 0.876
Miss      ('What time do you close on Sundays?'): L2 = 1.448

Proposed threshold: reject if best L2 > 1.162

(L2=0.851) accept  ✅ correct            How do I start a return?
(L2=0.798) accept  ✅ correct            Do you have a mobile app?
(L2=1.068) accept  ✅ correct            Is there a loyalty program?
(L2=0.736) accept  ✅ correct            When was the company founded?
(L2=1.109) accept  ✅ correct            Refund?
(L2=1.122) accept  ✅ correct            How can I reach support at night?
(L2=1.311) reject  ✅ correct            Who is the CEO?
(L2=1.722) reject  ✅ correct            What is the weather in Paris?
(L2=1.448) reject  ✅ correct            What time do you close on Sundays?


In [16]:
# part 3 experiment 3
import re

STOPWORDS = {"what", "is", "your", "the", "a", "an", "do", "you", "are", "how",
             "can", "i", "my", "to", "of", "for", "on", "in", "and", "or", "does"}

def tokens(text):
    return set(re.findall(r"[a-z0-9']+", text.lower())) - STOPWORDS

def keyword_search(query, k=2):
    """Naive keyword baseline: rank corpus pieces by number of shared (non-stopword) words."""
    q = tokens(query)
    scored = sorted(((len(q & tokens(doc)), i) for i, doc in enumerate(corpus)),
                    key=lambda t: (-t[0], t[1]))
    return [(s, corpus[i]) for s, i in scored[:k]]

pairs = [
    ("What is your return policy?", "wat is ur retrun polcy"),
    ("Do you take PayPal?",         "do u tke paypl"),
    ("How long is the warranty?",   "how lng is teh warrenty"),
]

for clean, typo in pairs:
    print(f"\n{'=' * 70}\nCLEAN: {clean}\nTYPO:  {typo}\n{'=' * 70}")

    sem_clean = retrieve_similar(clean, k=3)
    sem_typo  = retrieve_similar(typo, k=3)
    overlap = {r["index"] for r in sem_clean} & {r["index"] for r in sem_typo}
    print(f"\nSemantic search: {len(overlap)}/3 of the clean query's top-3 also appear in the typo query's top-3")
    print(f"  top hit (clean): (L2={sem_clean[0]['score']:.3f}) {sem_clean[0]['document'][:70]}")
    print(f"  top hit (typo):  (L2={sem_typo[0]['score']:.3f}) {sem_typo[0]['document'][:70]}")

    kw_clean = keyword_search(clean, k=1)[0]
    kw_typo  = keyword_search(typo, k=1)[0]
    print("\nKeyword search:")
    print(f"  top hit (clean): (shared words={kw_clean[0]}) {kw_clean[1][:70]}")
    if kw_typo[0] == 0:
        print("  top hit (typo):  (shared words=0) NO MATCH, nothing to retrieve")
    else:
        print(f"  top hit (typo):  (shared words={kw_typo[0]}) {kw_typo[1][:70]}")


CLEAN: What is your return policy?
TYPO:  wat is ur retrun polcy

Semantic search: 1/3 of the clean query's top-3 also appear in the typo query's top-3
  top hit (clean): (L2=0.906) Returns must be initiated within 14 days of delivery.
  top hit (typo):  (L2=1.781) The moon is made of green cheese.

Keyword search:
  top hit (clean): (shared words=2) TechGadgets Inc. Customer Policy Handbook. Returns and Refunds. Custom
  top hit (typo):  (shared words=0) NO MATCH, nothing to retrieve

CLEAN: Do you take PayPal?
TYPO:  do u tke paypl

Semantic search: 1/3 of the clean query's top-3 also appear in the typo query's top-3
  top hit (clean): (L2=0.876) We accept Visa, Mastercard, American Express, and PayPal.
  top hit (typo):  (L2=1.515) We accept Visa, Mastercard, American Express, and PayPal.

Keyword search:
  top hit (clean): (shared words=1) We accept Visa, Mastercard, American Express, and PayPal.
  top hit (typo):  (shared words=0) NO MATCH, nothing to retrieve

CLEAN: How long is

## Part 4: Generation with RAG (10 pts)

Now the full pipeline: retrieve → build an augmented prompt → generate. flan-t5-base
is instruction-tuned, so "answer using this context" actually works — and because it's
small and knows nothing about our fictional company, the **difference between RAG and
no-RAG will be dramatic** (that's the point of the comparison).


In [17]:
def rag_answer(question, k=3):
    """Answer a question with retrieval-augmented generation."""
    retrieved = retrieve_similar(question, k)
    context = "\n".join(r["document"] for r in retrieved)
    prompt = (f"Answer the question using only the context below.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}\nAnswer:")
    answer = generator(prompt, max_new_tokens=80)[0]["generated_text"]
    return {"question": question, "context": context,
            "retrieved_docs": retrieved, "answer": answer}

test_questions = [
    "How long do I have to return a product?",
    "What payment methods do you accept?",
    "How much does the premium subscription cost?",
    "Is there a student discount available?",
]
for q in test_questions:
    r = rag_answer(q)
    print(f"\nQ: {q}\nA: {r['answer']}")


Q: How long do I have to return a product?
A: 14 days

Q: What payment methods do you accept?
A: Visa, Mastercard, American Express, and PayPal

Q: How much does the premium subscription cost?
A: $29.99

Q: Is there a student discount available?
A: Yes


In [18]:
# The comparison that justifies the whole architecture: RAG vs the same LLM alone.
def compare_approaches(question):
    rag = rag_answer(question)
    pure = generator(f"Question: {question}\nAnswer:", max_new_tokens=80)[0]["generated_text"]
    return {"question": question, "rag_answer": rag["answer"], "pure_llm_answer": pure}

for q in ["What is your return policy?",
          "How much does premium cost?",
          "Do you ship internationally?"]:
    r = compare_approaches(q)
    print(f"\nQ: {r['question']}")
    print(f"  [RAG]      {r['rag_answer']}")
    print(f"  [Pure LLM] {r['pure_llm_answer']}")
    print("-" * 60)


Q: What is your return policy?
  [RAG]      Customers may return most items within 30 days of delivery for a full refund
  [Pure LLM] a one-year warranty
------------------------------------------------------------

Q: How much does premium cost?
  [RAG]      $29.99
  [Pure LLM] $.25
------------------------------------------------------------

Q: Do you ship internationally?
  [RAG]      Yes
  [Pure LLM] no
------------------------------------------------------------


### 🧪 Experiment

- Shrink the chunks to `chunk_size=30, overlap=0`, rebuild `corpus` + the FAISS index (re-run the Part 2 cells), and re-ask the four test questions — watch retrieval fall apart on multi-fact questions like the warranty one.
- Ask a question the corpus can't answer ("Who is the CEO?") through BOTH `rag_answer` and the bare generator. Does RAG admit ignorance better than the bare LLM, or does it hallucinate from irrelevant context?
- Bump `k=3` to `k=8` in `rag_answer`. Does more context help, or does the extra noise drown the model? Small models are especially easy to drown.


In [19]:
# part 4 experiment 1

orig_corpus, orig_index, orig_embeddings = corpus, index, corpus_embeddings

def build_kb(chunk_size, overlap):
    """Rebuild corpus + FAISS index (retrieve_similar reads these globals)."""
    global corpus, index, corpus_embeddings
    new_chunks = chunk_text(policy_document, chunk_size=chunk_size, overlap=overlap)
    corpus = documents + new_chunks
    corpus_embeddings = embedding_model.encode(corpus)
    index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
    index.add(corpus_embeddings.astype("float32"))
    return len(new_chunks)

qs = test_questions + ["What does the warranty not cover?"]   # the warranty question is the multi-fact one

def ask_all(label):
    print(f"\n{'=' * 70}\n{label}\n{'=' * 70}")
    for q in qs:
        r = rag_answer(q, k=3)
        print(f"\nQ: {q}\nA: {r['answer']}")
        print("   retrieved:", " | ".join(d["document"][:40] for d in r["retrieved_docs"]))

try:
    ask_all("BASELINE: chunk_size=60, overlap=15")
    n = build_kb(30, 0)
    ask_all(f"SMALL CHUNKS: chunk_size=30, overlap=0 ({n} policy chunks)")
finally:
    corpus, index, corpus_embeddings = orig_corpus, orig_index, orig_embeddings
    print(f"\n✅ Restored original knowledge base ({len(corpus)} pieces)")


BASELINE: chunk_size=60, overlap=15

Q: How long do I have to return a product?
A: 14 days
   retrieved: Returns must be initiated within 14 days | TechGadgets Inc. Customer Policy Handboo | We offer a 30-day money-back guarantee o

Q: What payment methods do you accept?
A: Visa, Mastercard, American Express, and PayPal
   retrieved: We accept Visa, Mastercard, American Exp | We offer a 30-day money-back guarantee o | within 5-7 business days after we receiv

Q: How much does the premium subscription cost?
A: $29.99
   retrieved: Our premium subscription costs $29.99 pe | for 1 to 3 additional years. Warranty cl | sales, and free express shipping. Enterp

Q: Is there a student discount available?
A: Yes
   retrieved: Student discounts of 20% are available w | for 1 to 3 additional years. Warranty cl | Our premium subscription costs $29.99 pe

Q: What does the warranty not cover?
A: accidental damage, water damage, or normal wear
   retrieved: customer's responsibility. Orders placed |

In [20]:
# part 4 experiment 2

unanswerable = ["Who is the CEO?",
                "What time do you close on Sundays?",
                "What is the weather in Paris?"]

for q in unanswerable:
    r = rag_answer(q, k=3)
    pure = generator(f"Question: {q}\nAnswer:", max_new_tokens=80)[0]["generated_text"]

    # Extra: same retrieval, but the prompt explicitly allows "I don't know"
    guarded_prompt = (f"Answer the question using only the context below. "
                      f"If the answer is not in the context, say \"I don't know\".\n\n"
                      f"Context:\n{r['context']}\n\nQuestion: {q}\nAnswer:")
    guarded = generator(guarded_prompt, max_new_tokens=80)[0]["generated_text"]

    print(f"\nQ: {q}   (best L2 = {r['retrieved_docs'][0]['score']:.3f})")
    print(f"  top context:  {r['retrieved_docs'][0]['document'][:70]}")
    print(f"  [RAG]         {r['answer']}")
    print(f"  [Pure LLM]    {pure}")
    print(f"  [RAG + 'I don't know' prompt] {guarded}")


Q: Who is the CEO?   (best L2 = 1.311)
  top context:  Our company was founded in 2010 and is headquartered in San Francisco.
  [RAG]         San Francisco
  [Pure LLM]    john mccain
  [RAG + 'I don't know' prompt] I don't know

Q: What time do you close on Sundays?   (best L2 = 1.448)
  top context:  Returns must be initiated within 14 days of delivery.
  [RAG]         24/7
  [Pure LLM]    midnight
  [RAG + 'I don't know' prompt] I don't know

Q: What is the weather in Paris?   (best L2 = 1.722)
  top context:  The moon is made of green cheese.
  [RAG]         rainy
  [Pure LLM]    rainy
  [RAG + 'I don't know' prompt] I don't know


In [21]:
# part 4 experiment 3

for q in qs:
    print(f"\nQ: {q}")
    for k in [3, 8]:
        r = rag_answer(q, k=k)
        prompt = (f"Answer the question using only the context below.\n\n"
                  f"Context:\n{r['context']}\n\nQuestion: {q}\nAnswer:")
        n_tokens = len(_tok(prompt).input_ids)
        flag = "  ⚠️ over 512, the end of the prompt (the question) gets cut off" if n_tokens > 512 else ""
        print(f"  k={k}: {r['answer']!r}  [{n_tokens} prompt tokens{flag}]")


Q: How long do I have to return a product?
  k=3: '14 days'  [136 prompt tokens]
  k=8: '14 days'  [388 prompt tokens]

Q: What payment methods do you accept?
  k=3: 'Visa, Mastercard, American Express, and PayPal'  [126 prompt tokens]
  k=8: 'Visa, Mastercard, American Express, and PayPal'  [270 prompt tokens]

Q: How much does the premium subscription cost?
  k=3: '$29.99'  [149 prompt tokens]
  k=8: '$29.99'  [331 prompt tokens]

Q: Is there a student discount available?
  k=3: 'Yes'  [121 prompt tokens]
  k=8: 'Yes'  [274 prompt tokens]

Q: What does the warranty not cover?
  k=3: 'accidental damage, water damage, or normal wear'  [186 prompt tokens]
  k=8: 'accidental damage, water damage, or normal wear'  [346 prompt tokens]


**Summarize in a markdown cell (2–3 rows are enough):** a small comparison table —
question | RAG answer | pure-LLM answer | which is right? The pure LLM either refuses,
hallucinates a generic policy, or guesses — it has never seen *your* documents. That
gap is the business case for RAG. Save your best example for **Q2**.


# Answer

### RAG vs. pure LLM (flan-t5-base)( reference cell 31)

| Question | RAG answer | Pure-LLM answer | Which is right? |
|---|---|---|---|
| What is your return policy? | Customers may return most items within 30 days of delivery for a full refund | a one-year warranty | **RAG.** It matches the handbook, though it is incomplete (electronics have a shorter 14-day window). The pure LLM answers a different question. |
| How much does premium cost? | $29.99 | $.25 | **RAG.** The FAQ and handbook both say $29.99 per month. The pure LLM invented a price. |
| Do you ship internationally? | Yes | no | **RAG.** The corpus says we ship to over 50 countries. The pure LLM's "no" is a guess that happens to be wrong. |

The pure LLM has never seen TechGadgets' documents, so it guesses: it answered a different question (a warranty for a return policy), made up a price, and gave a confident "no" that contradicts the policy. RAG retrieves the relevant passage and the model reads the answer from it. That gap is the business case for RAG.

In [22]:
# ── OPTIONAL (not required, $0 stays $0): swap in a frontier model as the generator ──
# ⚠️ Requires creating a free external account + API key — skip freely.
# The retrieval half stays identical; only generation changes. Two free options:
#
# Option 1 — OpenRouter free tier (sign up free at openrouter.ai, no card needed;
#            save the key as Colab secret OPENROUTER_API_KEY):
# from google.colab import userdata
# from openai import OpenAI
# client = OpenAI(base_url="https://openrouter.ai/api/v1",
#                 api_key=userdata.get("OPENROUTER_API_KEY"))
# def rag_answer_frontier(question, k=3):
#     ctx = "\n".join(r["document"] for r in retrieve_similar(question, k))
#     msg = f"Answer using only this context:\n{ctx}\n\nQuestion: {question}"
#     resp = client.chat.completions.create(
#         model="openai/gpt-oss-20b:free",   # free ids rotate — check openrouter.ai/models
#         messages=[{"role": "user", "content": msg}])
#     return resp.choices[0].message.content
#
# Option 2 — Gemini free tier (key from Google AI Studio, save as Colab secret GOOGLE_API_KEY):
# see the course setup guide; same pattern, different client.
print("Optional cell — skip freely, flan-t5 is enough for full credit.")

Optional cell — skip freely, flan-t5 is enough for full credit.


## Part 5: Questions (6 pts)

Answer **Q1–Q4** (listed in the Submit section below) in a markdown cell here. Use
concrete evidence from your runs: your chunk-size experiments (Q1), your best
RAG-vs-LLM example (Q2), your retrieval failure (Q3), and a deployment sketch —
where do the documents live, what updates the index, what does it cost (Q4).


## 🎮 Your Domain Playground

The demo taught you the METHOD on a fictional company's documents. Now build RAG over
a knowledge base from **a domain you're passionate about** — a world where YOU are the
expert and can instantly tell a right answer from a confident wrong one. Ideas:

- your industry's public FAQ or policy pages
- a game's wiki pages (build rules, item stats, lore)
- your fantasy league's rulebook or your team's bylaws
- a product manual, a gym's class policies, a band's tour FAQ

**Steps:**

1. **Collect 10–15 passages** (a sentence to a paragraph each) from your domain and
   paste them into `my_documents` below. Longer source? Paste it as one string and
   `chunk_text` it — that's what Part 1 was for.


In [29]:
# Paste 10-15 passages from YOUR domain, then run this cell.

# Long source document instead? Uncomment:
my_long_doc = """How To Win

1) Survival Victory
You win the instant your opponent has no life cards in his Life Deck. Cards are taken from
the Life Deck by:

*Discarding and drawing cards in the course of play.

*Dealing Life Cards of Damage. Your opponent discards one card from his Life Deck for
each Life Card of damage.

*Dealing Power Stages of Damage. Your opponent discards one card from his Life Deck for
each power stage of damage taken by a Personality when that Personality is at their lowest power
stage.

*Card Effects. Some cards have effects that result in cards being discarded from your
opponent’s Life Deck. Such effects are not considered to be damage even though life cards
get discarded.

2) Dragon Ball Victory
You win the game the moment you control all 7 Dragon Balls of the same set. The challenge
is keeping them under your control, because they can be captured in various ways.

However, if you control 6 Dragon Balls and your opponent controls the last Dragon Ball in the
set, if you capture that Dragon Ball from your opponent, you do not win the game until the
start of your next turn.

3) Most Powerful Personality Victory
When your Main Personality reaches the highest level possible out of all personalities in the
current game, if you reach that level by Anger without having a card effect used against you 
to postpone your victory or a rule preventing you winning the game, you win thegame. If a 
card is used against you to postpone your victory, when that card effect ends if you are
still at the highest personality level possible still, you win the game.
"""
dbz_chunk_size, dbz_overlap = 60, 15     # see note below

my_documents = chunk_text(my_long_doc, chunk_size=dbz_chunk_size, overlap=dbz_overlap)  # "=" not "+", so re-runs are safe

corpus = my_documents
corpus_embeddings = embedding_model.encode(corpus)
index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
index.add(corpus_embeddings.astype("float32"))
print(f"✅ Rebuilt the vector store over YOUR domain: {index.ntotal} pieces")

✅ Rebuilt the vector store over YOUR domain: 6 pieces


In [30]:
# Playground: re-ask through the same pipeline
# (question, what the rulebook text says, so you can grade as the expert)
dbz_questions = [
    ("How many Dragon Balls do I need to win?",
     "all 7 Dragon Balls of the same set"),
    ("I control 6 Dragon Balls and I capture the last one from my opponent. When do I win?",
     "not until the start of your next turn"),
    ("Does discarding cards from my opponent's Life Deck with a card effect count as damage?",
     "no, such effects are not considered damage even though life cards get discarded"),
    ("What are the ways to win the game?",
     "Survival, Dragon Ball, and Most Powerful Personality victory"),
    ("When does a Personality's power stage damage make my opponent discard Life Deck cards?",
     "one card per power stage of damage, when the Personality is at their lowest power stage"),
    ("What is the maximum hand size?",
     "NOT IN THE TEXT"),
]

for q, truth in dbz_questions:
    r = rag_answer(q, k=3)
    print(f"\nQ: {q}")
    print(f"  Rulebook says: {truth}")
    print(f"  [RAG answer]:  {r['answer']}")
    for d in r["retrieved_docs"]:
        print(f"     (L2={d['score']:.3f}, chunk {d['index']}) {d['document'][:70]}")


Q: How many Dragon Balls do I need to win?
  Rulebook says: all 7 Dragon Balls of the same set
  [RAG answer]:  6
     (L2=0.684, chunk 3) Dragon Balls of the same set. The challenge is keeping them under your
     (L2=0.816, chunk 2) *Card Effects. Some cards have effects that result in cards being disc
     (L2=1.275, chunk 4) Ball from your opponent, you do not win the game until the start of yo

Q: I control 6 Dragon Balls and I capture the last one from my opponent. When do I win?
  Rulebook says: not until the start of your next turn
  [RAG answer]:  the start of your next turn
     (L2=0.463, chunk 3) Dragon Balls of the same set. The challenge is keeping them under your
     (L2=0.712, chunk 2) *Card Effects. Some cards have effects that result in cards being disc
     (L2=1.007, chunk 4) Ball from your opponent, you do not win the game until the start of yo

Q: Does discarding cards from my opponent's Life Deck with a card effect count as damage?
  Rulebook says: no, such eff

In [31]:
# Playground: RAG vs pure LLM on the best question
best_q = "How many Dragon Balls do I need to win?"   # swap in your best example after Cell 2

r = compare_approaches(best_q)
print(f"Q: {r['question']}")
print(f"  [RAG]      {r['rag_answer']}")
print(f"  [Pure LLM] {r['pure_llm_answer']}")

Q: How many Dragon Balls do I need to win?
  [RAG]      6
  [Pure LLM] ten


Wrong answer, testing different question.

In [32]:
# Playground: RAG vs pure LLM on the best question
best_q = "I control 6 Dragon Balls, and I capture the last one from my opponent, when do I win?"   # swap in your best example after Cell 2

r = compare_approaches(best_q)
print(f"Q: {r['question']}")
print(f"  [RAG]      {r['rag_answer']}")
print(f"  [Pure LLM] {r['pure_llm_answer']}")

Q: I control 6 Dragon Balls, and I capture the last one from my opponent, when do I win?
  [RAG]      the start of your *Card Effects
  [Pure LLM] i win the game


RAG is correct answer

In [33]:
# Playground: which questions need bigger chunks or a different k?
def build_dbz_kb(chunk_size, overlap):
    global corpus, index, corpus_embeddings
    corpus = chunk_text(my_long_doc, chunk_size=chunk_size, overlap=overlap)
    corpus_embeddings = embedding_model.encode(corpus)
    index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
    index.add(corpus_embeddings.astype("float32"))
    return len(corpus)

# (question, phrases that must ALL appear in the retrieved context for the answer to be possible)
hard_qs = [
    ("What are the ways to win the game?",
     ["Survival Victory", "Dragon Ball Victory", "Most Powerful Personality"]),
    ("I control 6 Dragon Balls and I capture the last one from my opponent. When do I win?",
     ["start of your next turn"]),
]

try:
    for size, ov in [(30, 5), (40, 10), (60, 15), (100, 20)]:
        n = build_dbz_kb(size, ov)
        print(f"\n{'=' * 70}\nchunk_size={size}, overlap={ov} -> {n} chunks\n{'=' * 70}")
        for q, must_have in hard_qs:
            for k in [3, 5]:
                r = rag_answer(q, k=k)
                has_it = all(p.lower() in r["context"].lower() for p in must_have)
                print(f"  k={k} | context has the facts: {'✅' if has_it else '❌'} | {q[:45]}...")
                print(f"        answer: {r['answer']!r}")
finally:
    build_dbz_kb(dbz_chunk_size, dbz_overlap)   # restore the setting from Cell 1
    print(f"\n✅ Restored {dbz_chunk_size}/{dbz_overlap} knowledge base ({len(corpus)} pieces)")


chunk_size=30, overlap=5 -> 11 chunks
  k=3 | context has the facts: ❌ | What are the ways to win the game?...
        answer: 'Survival Victory You win the instant your opponent has no life cards in his Life Deck. Cards are taken from the Life Deck by: *Discarding and if you capture that Dragon Ball from your opponent, you do not win the game until the start of your next turn.'
  k=5 | context has the facts: ❌ | What are the ways to win the game?...
        answer: 'Survival Victory You win the instant your opponent has no life cards in his Life Deck. Cards are taken from the Life Deck by: *Discarding and if you capture that Dragon Ball from your opponent, you do not win the game until the start of your next turn.'
  k=3 | context has the facts: ✅ | I control 6 Dragon Balls and I capture the la...
        answer: 'the start of your next turn'
  k=5 | context has the facts: ✅ | I control 6 Dragon Balls and I capture the la...
        answer: 'the start of your next turn'

chunk_size=4

2. **Re-ask through the same pipeline:** call `rag_answer()` on 4–5 questions a
   real fan/insider/customer of your domain would ask, plus one it can't answer. Run
   `compare_approaches()` on your best one.
3. **The insider test (markdown cell, 4–6 sentences):** as the domain expert, grade the
   answers — right, subtly wrong, or confidently hallucinated? Which questions needed
   bigger chunks or a different k than the demo's settings? An answer that would fool
   an outsider but not you is exactly the failure mode businesses pay experts to catch —
   name one if you found it.


# Insider Test

## Grading

Grading the six answers against the rulebook passage, three answers were right; the timing after capturing the 6th Dragon Ball, "no" for card-effect discards, and the power-stage trigger, and three were wrong. The subtle one is "How many Dragon Balls do I need to win?" → "6": the rule is 7, but the top-ranked chunk contains "if you control 6 Dragon Balls," and the model grabbed that number even though the chunk with "all 7" was also retrieved. A newcomer would believe it, but a player would not. This kind of confusion would definetly be clarified by the ref in live play.  "What are the ways to win?" returned a fragment of the Survival Victory bullet list, even though at chunk_size=60, k=3 the context contained all three victory conditions, so this is a generation weakness of flan-t5-base, likey due to coping instead of summarizing. "What is the maximum hand size?" is not in the text, yet the system answered "Dragon Ball Victory" from irrelevant context at L2 ≈ 1.8 versus 0.41-0.94 for good hits, so a distance threshold would have caught it. The multi-part question needed bigger chunks or more context: it failed at 30/5 (both k values) and at 40/10 and 100/20 with k=3, and only had all three conditions in context at k=3 with 60/15, while the single-fact capture question worked at every setting.

# HW Questions and Answers
- **Q1:** How did you decide on chunk size and overlap?

- **A1:** I determined chunk size by context. Trying to set the chunk bounds to the start and end of a rule was the goal. I test 4 different types of chunking and found that a chunk of 60 with a 15 word overlap semed the best. I chose the 15 word overlap since that is like the average length of a sentence.


- **Q2:** Show an example where RAG improved the LLM's answer.

- **A2:** The best example would be "I control 6 Dragon Balls and I capture the last one from my opponent. When do I win?" The rulebook says you do not win until the start of your next turn. RAG retrieved the chunk containing that rule (its closest match, L2 = 0.463) and answered "the start of your next turn," which is correct. The same flan-t5-base model with no retrieval answered: [PASTE the pure-LLM answer from cell 44]. The pure model has never seen this game's rules, so it can only guess: when I asked it "How many Dragon Balls do I need to win?" it answered "ten," while the rule is 7. 


- **Q3:** Show an example where retrieval failed. Why?

- **A3:** The clearest failure is a question the rulebook cannot answer: "What is the maximum hand size?" FAISS always returns the nearest chunks, and here the nearest was only L2 = 1.802 compared with 0.41 to 0.94 for every answerable question, but the system still answered "Dragon Ball Victory" from unrelated text. Nothing in the pipeline checks whether the retrieved context is actually relevant, and the generator answers anyway. This is why I think retrival failed.


- **Q4:** How would you deploy this for a real business application?

- **A4:** For a production applicaiton this should not be a notebook. I would have module python scripts. One for ingestion, chunking, embedding, vector db, retrieval, and prompts. This modularatiy make it easier to scalce, debug and deploy via a container like docker. Then there should be gaurd rails for when RAg fails to actually know the answer. A bad response in a business is like a bad rumor of layoffs after an earnings call.

## 🤔 Debrief

1. Your FAISS search *always* returns k results, relevant or not — and the LLM will
   confidently answer from bad context. In production, what guardrails stop a customer
   seeing an answer built on garbage retrieval?
2. When is RAG the WRONG tool? (Think: data that changes every second, questions
   needing reasoning across ALL documents at once, or 20 documents you could just
   paste into one long-context prompt.)


## 📤 Submit (HW7 — 40 points)

**Deliverables (submit through Canvas):**
- Completed `.ipynb` executed top-to-bottom, INCLUDING the 🎮 Playground run on documents from your own domain (built yourself — no NotebookLM or other pre-built RAG services)
- A short RAG vs pure-LLM comparison table (markdown cell)
- Your insider-test writeup grading the answers on your domain's questions

**Answer these questions in a markdown cell at the end of your notebook:**
- **Q1:** How did you decide on chunk size and overlap?
- **Q2:** Show an example where RAG improved the LLM's answer.
- **Q3:** Show an example where retrieval failed. Why?
- **Q4:** How would you deploy this for a real business application?

**Grading:**

| Part | Points |
|---|---|
| Part 1: Document Chunking | 6 |
| Part 2: Embedding Index | 8 |
| Part 3: Retrieval | 10 |
| Part 4: Generation with RAG | 10 |
| Part 5: Questions | 6 |

**Due Sunday 10:00 p.m.** — see the Canvas assignment for the exact date. Late work is not accepted (see syllabus).

*AI policy: AI tools are allowed and encouraged. Add one line acknowledging what you used, and be ready to explain every part of what you submit.*
